In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy

TIME_OFFSET = 10800 #ADT to UTC is 3hrs

In [14]:
def entropy_feature(x):
    counts = x.value_counts()
    if len(counts) <= 1:
        return 0.0
    return entropy(counts)

def coeff_var(x):
    mean = x.mean()
    if mean == 0:
        return 0.0
    return x.std() / mean

In [15]:
def process_file(k, input_csv, output_csv):
    print(f"Processing {input_csv}...")

    df = pd.read_csv(input_csv)

    df["timestamp"] = df["frame.time_epoch"] + TIME_OFFSET

    # Sort for IAT calculation
    df = df.sort_values("timestamp")

    df["time_window"] = (df["timestamp"]*k).astype(int) #k tweaks the time window size 
    packet_counts = df.groupby("time_window").size()
    # print(packet_counts)
    print("description:")
    print(packet_counts.describe())

    df["iat"] = df.groupby(["ip.src", "ip.dst"])["timestamp"].diff().fillna(0) #this basically finds the inter-arrival time for src-dst pairs

    has_transaction_id = "modbus.transaction_id" in df.columns

    agg_dict = {
        # Existing
        "packet_count": ("frame.len", "count"),
        "total_bytes": ("frame.len", "sum"),
        "mean_packet_size": ("frame.len", "mean"),
        "std_packet_size": ("frame.len", "std"),
    
        # NEW: packet size extremes
        "min_packet_size": ("frame.len", "min"),
        "max_packet_size": ("frame.len", "max"),
    
        # Timing
        "iat_mean": ("iat", "mean"),
        "iat_std": ("iat", "std"),
        "min_iat": ("iat", "min"),
        "max_iat": ("iat", "max"),
    
        # NEW: periodicity
        "iat_cv": ("iat", coeff_var),
    
        # Function codes
        "unique_func_codes": ("modbus.func_code", "nunique"),
    
        # NEW: entropy
        "func_entropy": ("modbus.func_code", entropy_feature),
    
        "read_count": ("modbus.func_code", lambda x: x.isin([3, 4]).sum()),
        "write_count": ("modbus.func_code", lambda x: x.isin([5, 6, 15, 16]).sum()),
    
        # NEW: individual function counts
        "fc_3_count": ("modbus.func_code", lambda x: (x == 3).sum()),
        "fc_4_count": ("modbus.func_code", lambda x: (x == 4).sum()),
        "fc_6_count": ("modbus.func_code", lambda x: (x == 6).sum()),
        "fc_16_count": ("modbus.func_code", lambda x: (x == 16).sum()),
    
        # Exception
        "exception_count": ("modbus.exception_code", lambda x: x.notna().sum()),
    
        # NEW: register features
        "unique_registers": ("modbus.reference_num", "nunique"),
        "register_mean": ("modbus.reference_num", "mean"),
        "register_std": ("modbus.reference_num", "std"),
        "register_entropy": ("modbus.reference_num", entropy_feature),

        # Response timing
        "response_time_mean": ("modbus.response_time", "mean"),
        "response_time_std": ("modbus.response_time", "std"),

        # Register counts
        "unique_read_registers": ("modbus.read_reference_num", "nunique"),
        "unique_write_registers": ("modbus.write_reference_num", "nunique"),

        # Word count
        "word_cnt_mean": ("modbus.word_cnt", "mean"),
        "word_cnt_std": ("modbus.word_cnt", "std"),

        # Response size
        "byte_cnt_mean": ("modbus.byte_cnt", "mean"),

        # Register value behavior
        "regval_mean": ("modbus.regval_uint16", "mean"),
        "regval_std": ("modbus.regval_uint16", "std"),
    
        # NEW: burstiness
        "burstiness": ("iat", lambda x: x.max() - x.min())
    }

    if has_transaction_id:
        agg_dict["unique_transaction_ids"] = ("modbus.transaction_id", "nunique")

    flows = df.groupby(
        ["time_window", "ip.src", "ip.dst"]
    ).agg(**agg_dict).reset_index()

    flows = flows.fillna(0)
    flows["write_ratio"] = np.where(flows["packet_count"] > 0, flows["write_count"] / flows["packet_count"], 0.0)
    if has_transaction_id:
        flows["duplicate_transaction_ids"] = flows["packet_count"] - flows["unique_transaction_ids"]
        flows["transaction_id_ratio"] = np.where(flows["packet_count"] > 0, flows["unique_transaction_ids"] / flows["packet_count"], 0.0)
    else:
        flows["unique_transaction_ids"] = 0
        flows["duplicate_transaction_ids"] = 0
        flows["transaction_id_ratio"] = 0.0
    
    flows["write_ratio"] = np.where(
        flows["packet_count"] > 0,
        flows["write_count"] / flows["packet_count"],
        0.0
    )
    
    flows["exception_ratio"] = np.where(
        flows["packet_count"] > 0,
        flows["exception_count"] / flows["packet_count"],
        0.0
    )
    
    # NEW: read ratio
    flows["read_ratio"] = np.where(
        flows["packet_count"] > 0,
        flows["read_count"] / flows["packet_count"],
        0.0
    )

    flows["regval_delta"] = flows.groupby(["ip.src", "ip.dst"])["regval_mean"].diff().fillna(0)

    # packets_per_window = flows.groupby("time_window")["packet_count"].sum().sort_index()

    # plt.figure()
    # plt.plot(packets_per_window.index - packets_per_window.index.min(), packets_per_window.values)
    # plt.xlabel(f"{1000/k:.2f} ms windows")
    # plt.ylabel("Total Packets")
    # plt.title("Packets per Time Window")
    # plt.show()

    flows.to_csv(output_csv, index=False)
    print(f"Saved → {output_csv}\n")

In [16]:
process_file(1, "../train/benign_nw.csv", "../train/1s_benign_flows.csv")

Processing ../train/benign_nw.csv...
description:
count    7085.000000
mean       75.892025
std        90.031639
min         1.000000
25%        50.000000
50%        50.000000
75%        90.000000
max      1292.000000
dtype: float64
Saved → ../train/1s_benign_flows.csv



In [17]:
process_file(1, "../train/cscada_attack_ssw.csv", "../train/1s_cscada_flows.csv")

Processing ../train/cscada_attack_ssw.csv...
description:
count    14981.000000
mean        77.228289
std        144.291295
min          1.000000
25%         45.000000
50%         50.000000
75%         90.000000
max       8241.000000
dtype: float64
Saved → ../train/1s_cscada_flows.csv



In [18]:
process_file(1, "../train/ext_attack_nw.csv", "../train/1s_external_flows.csv")

Processing ../train/ext_attack_nw.csv...
description:
count     1881.000000
mean       290.755981
std       1252.626481
min          1.000000
25%         45.000000
50%         45.000000
75%         51.000000
max      20281.000000
dtype: float64
Saved → ../train/1s_external_flows.csv



In [24]:
df = pd.read_csv("../train/1s_benign_flows.csv")
print(df.columns)

Index(['time_window', 'ip.src', 'ip.dst', 'packet_count', 'total_bytes',
       'mean_packet_size', 'std_packet_size', 'min_packet_size',
       'max_packet_size', 'iat_mean', 'iat_std', 'min_iat', 'max_iat',
       'iat_cv', 'unique_func_codes', 'func_entropy', 'read_count',
       'write_count', 'fc_3_count', 'fc_4_count', 'fc_6_count', 'fc_16_count',
       'exception_count', 'unique_registers', 'register_mean', 'register_std',
       'register_entropy', 'response_time_mean', 'response_time_std',
       'unique_read_registers', 'unique_write_registers', 'word_cnt_mean',
       'word_cnt_std', 'byte_cnt_mean', 'regval_mean', 'regval_std',
       'burstiness', 'write_ratio', 'unique_transaction_ids',
       'duplicate_transaction_ids', 'transaction_id_ratio', 'exception_ratio',
       'read_ratio', 'regval_delta'],
      dtype='str')


In [19]:
# process_file(10, "../train/benign_nw.csv", "../train/100ms_benign_flows.csv") #this is for one second windows

In [20]:
# process_file(10, "../train/cscada_attack_ssw.csv", "../train/100ms_cscada_flows.csv")

In [21]:
# process_file(10, "../train/ext_attack_nw.csv", "../train/100ms_external_flows.csv")